# MATH1604 — Analysis of Python Quiz Responses
## Review Notebook: Team Member 1

**Author:** Abdulaziz Aldawsari  
**Student ID:** 202003156  
**Role:** Team Member 1 — Parsing Module  
**Module:** MATH1604 Modelling for Big Data  
**Date:** May 2025

---

## 1. Introduction

This notebook documents my individual contribution to the MATH1604 group project. The project involves analysing a dataset of 64 respondents' answers to a 100-question multiple-choice Python quiz, with the goal of detecting whether the quiz setter arranged the correct answers in a deliberate pattern.

As Team Member 1, my responsibility was to write `data_extraction_M1.py` — the parsing module that forms the foundation of the entire pipeline. Without this module, no other part of the project can function, as every downstream module depends on being able to read and interpret the raw quiz answer files.

My module provides two functions:
- **`extract_answers_sequence(file_path)`** — reads a raw quiz answer `.txt` file and returns a structured list of 100 integers
- **`write_answers_sequence(answers, n, destination_path)`** — saves that list to a text file for downstream use

---
## 2. Module Overview

### 2.1 How the raw answer files are structured

Each raw quiz file follows a consistent format. Every question block looks like this:

```
Question N. [question text]
[ ] Answer N.1
[x] Answer N.2     ← this means option 2 was selected
[ ] Answer N.3
[ ] Answer N.4
```

A selected answer is marked with `[x]`. An unanswered question has all four options marked `[ ]`.

### 2.2 How my parser works

My parser reads the file line by line. When it detects a line starting with `Question N.`, it knows a new question is beginning. It then collects the four option lines that follow, checks which one contains `[x]`, and records the position (1, 2, 3, or 4). If none contain `[x]`, it records 0 for unanswered.

This approach is robust because it does not rely on fixed line numbers — it detects questions and options by their content, so it works correctly even if there are blank lines or varying amounts of question text.

---
## 3. Setting Up

In [ ]:
import sys
import os

# Add scripts folder to path so we can import our module
sys.path.insert(0, os.path.join('..', 'scripts'))

# Import our M1 functions
from data_extraction_M1 import extract_answers_sequence, write_answers_sequence

print('data_extraction_M1 imported successfully.')

In [ ]:
# Define folder paths
data_folder = os.path.join('..', 'data')
output_folder = os.path.join('..', 'output')

os.makedirs(output_folder, exist_ok=True)

print(f'Data folder:   {data_folder}')
print(f'Output folder: {output_folder}')

# Check how many respondent files are available
available_files = sorted(
    [f for f in os.listdir(data_folder) if f.startswith('answers_respondent_')],
    key=lambda x: int(x.replace('answers_respondent_', '').replace('.txt', ''))
)
print(f'\nRespondent files available: {len(available_files)}')
print(f'First file: {available_files[0]}')
print(f'Last file:  {available_files[-1]}')

---
## 4. Demonstrating `extract_answers_sequence`

### 4.1 Basic usage

In [ ]:
# Test on respondent 1
file_path = os.path.join(data_folder, 'answers_respondent_1.txt')
answers_r1 = extract_answers_sequence(file_path)

print(f'Return type : {type(answers_r1)}')
print(f'Length      : {len(answers_r1)}  (expected: 100)')
print(f'First 20    : {answers_r1[:20]}')
print(f'Last 20     : {answers_r1[80:]}')
print(f'Unique values found: {sorted(set(answers_r1))}')

### 4.2 Checking for unanswered questions

In [ ]:
unanswered = [i + 1 for i, a in enumerate(answers_r1) if a == 0]

if unanswered:
    print(f'Respondent 1 left {len(unanswered)} question(s) unanswered.')
    print(f'Unanswered question numbers: {unanswered}')
else:
    print('Respondent 1 answered all 100 questions.')

### 4.3 Testing on multiple respondents

In [ ]:
# Extract sequences for the first 5 respondents and compare
print('First 10 answers for respondents 1-5:\n')

for n in range(1, 6):
    fp = os.path.join(data_folder, f'answers_respondent_{n}.txt')
    seq = extract_answers_sequence(fp)
    print(f'Respondent {n}: {seq[:10]}')

### 4.4 Answer distribution for respondent 1

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Count how many times each answer (1-4) was selected by respondent 1
counts = Counter(a for a in answers_r1 if a != 0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([1, 2, 3, 4],
       [counts.get(1, 0), counts.get(2, 0), counts.get(3, 0), counts.get(4, 0)],
       color='steelblue', edgecolor='white')
ax.axhline(25, color='red', linestyle='--', label='Expected if uniform (25)')
ax.set_title('Answer Option Frequency — Respondent 1')
ax.set_xlabel('Answer Option')
ax.set_ylabel('Number of Questions')
ax.set_xticks([1, 2, 3, 4])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Counts:', dict(counts))

---
## 5. Demonstrating `write_answers_sequence`

### 5.1 Basic usage

In [ ]:
# Write respondent 1's sequence to the output folder
write_answers_sequence(answers_r1, 1, output_folder)

# Verify the file was created
written_file = os.path.join(output_folder, 'answers_list_respondent_1.txt')
print(f'File exists: {os.path.exists(written_file)}')

# Read it back and check contents
with open(written_file, 'r') as f:
    lines = f.readlines()

print(f'Number of lines: {len(lines)}  (expected: 100)')
print(f'First 10 lines: {[l.strip() for l in lines[:10]]}')

### 5.2 Verifying the written file matches the extracted sequence

In [ ]:
# Read the written file back as integers
with open(written_file, 'r') as f:
    written_answers = [int(line.strip()) for line in f.readlines()]

# Check they match exactly
if written_answers == answers_r1:
    print('Verification PASSED — written file matches extracted sequence exactly.')
else:
    print('Verification FAILED — mismatch detected.')
    differences = [(i+1, answers_r1[i], written_answers[i])
                   for i in range(100) if answers_r1[i] != written_answers[i]]
    print(f'Differences at questions: {differences}')

---
## 6. Error Handling Tests

Good code should fail gracefully with clear error messages. Here I demonstrate all three error cases my module handles.

In [ ]:
# Test 1: FileNotFoundError — file path does not exist
print('Test 1: Non-existent file path')
print('-' * 40)
try:
    extract_answers_sequence('this_file_does_not_exist.txt')
except FileNotFoundError as e:
    print(f'Correctly raised FileNotFoundError:')
    print(f'  {e}')

In [ ]:
# Test 2: ValueError — answers list is wrong length
print('Test 2: Wrong length answers list')
print('-' * 40)
try:
    write_answers_sequence([1, 2, 3], 1, output_folder)
except ValueError as e:
    print(f'Correctly raised ValueError:')
    print(f'  {e}')

In [ ]:
# Test 3: TypeError — respondent ID is not a positive integer
print('Test 3: Invalid respondent ID')
print('-' * 40)
try:
    write_answers_sequence(answers_r1, -5, output_folder)
except TypeError as e:
    print(f'Correctly raised TypeError:')
    print(f'  {e}')

In [ ]:
# Test 4: FileNotFoundError — destination folder does not exist
print('Test 4: Non-existent destination folder')
print('-' * 40)
try:
    write_answers_sequence(answers_r1, 1, 'folder_that_does_not_exist')
except FileNotFoundError as e:
    print(f'Correctly raised FileNotFoundError:')
    print(f'  {e}')

---
## 7. Processing All 64 Respondents

In [ ]:
# Extract and write sequences for all respondents
all_sequences = []
failed = []

for fname in available_files:
    fp = os.path.join(data_folder, fname)
    n = int(fname.replace('answers_respondent_', '').replace('.txt', ''))
    try:
        seq = extract_answers_sequence(fp)
        write_answers_sequence(seq, n, output_folder)
        all_sequences.append(seq)
    except Exception as e:
        print(f'Failed on {fname}: {e}')
        failed.append(fname)

print(f'\nSuccessfully processed: {len(all_sequences)} respondents')
print(f'Failed: {len(failed)}')

---
## 8. Limitations and Assumptions

**Assumption 1 — File encoding is UTF-8.**  
The parser opens files with `encoding='utf-8'`. If a file uses a different encoding, it may fail or produce incorrect results. This is unlikely given the files come from a controlled source, but worth noting.

**Assumption 2 — Exactly one `[x]` per question.**  
The parser assumes each answered question has exactly one `[x]`. If a file somehow contained two `[x]` marks for the same question, the parser would return the first one found. The `_get_selected_option` helper stops at the first match.

**Assumption 3 — Questions are always numbered sequentially.**  
The parser detects questions by matching `Question N.` at the start of a line. If any question text happened to start with this same pattern, it could be misidentified as a new question. This is highly unlikely in practice.

**Limitation 1 — No partial file recovery.**  
If a file contains fewer than 100 questions due to corruption or truncation, `extract_answers_sequence` raises a `ValueError` and the entire file is skipped. A more advanced implementation could return a partial sequence with 0s for missing questions.

**Limitation 2 — Output folder must already exist.**  
`write_answers_sequence` requires the destination folder to exist and raises `FileNotFoundError` if it does not. The pipeline script (`run_full_analysis_M4.py`) handles folder creation before calling this function, so in normal use this is not an issue.

---
## 9. Conclusion

*(Fill this in after running all cells and seeing the real results.)*

This notebook has demonstrated the full functionality of `data_extraction_M1.py`. Both functions work correctly:

- `extract_answers_sequence` successfully parses all [fill in: X] respondent files, returning lists of exactly 100 integers with values 1–4 or 0 for unanswered questions.
- `write_answers_sequence` correctly saves each sequence to a text file and the written output matches the extracted sequence exactly.
- All four error cases are handled with appropriate exception types and clear messages.

[Add 2–3 sentences about what you noticed from the answer distributions — did any respondent leave many questions unanswered? Did the answer options seem evenly distributed?]